# B05 · S3 — Motores de reglas (Pagarium)

**Objetivo (RA5-b):** pasar del micro-motor a los motores de producción: **DMN**, **GoRules ZEN** y `rule-engine`, y medir cuándo compensa.

> Práctica guiada de la S3 de los [apuntes](../apuntes.md).

## 1. Política de Pagarium como tabla DMN

| Importe | Antigüedad (meses) | Intentos | Decisión |
|---|---|---|---|
| `< 500` | `>= 6` | `< 4` | Aprobar |
| `< 500` | `>= 6` | `>= 4` | Revisar |
| `>= 500` | — | — | Revisar |
| — | `< 6` | — | Rechazar |

Hit policy propuesta: **unique** (una sola fila puede encajar).

## 2. `rule-engine` en Python puro

In [ ]:
%pip install rule-engine
import rule_engine

ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}
aprueba = rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos < 4")
rechaza = rule_engine.Rule("n_intentos >= 4 or antiguedad_meses < 6")
print("aprueba:", aprueba.matches(ctx))
print("rechaza:", rechaza.matches(ctx))

## 3. Actividad — añade la regla de "Revisar"

Implementa la tabla DMN completa con `rule-engine` (o con tu `MicroMotor2` de la S2) y pruébala con 4 casos que cubran las 4 filas.

In [ ]:
# TODO: implementa la politica completa y prueba 4 casos
...

## 4. GoRules ZEN (fintech)

El grafo JDM se define normalmente en el editor de GoRules y se ejecuta desde Python con `zen-engine`. La API cambia entre versiones: si el grafo no devuelve salida, exporta uno desde el editor.

In [ ]:
%pip install zen-engine
import json
from zen import ZenEngine

graph = {
    "contentType": "application/vnd.gorules.decision",
    "nodes": [
        {"id": "input", "type": "inputNode", "name": "Request", "position": {"x": 0, "y": 0}},
        {"id": "table", "type": "decisionTableNode", "name": "Politica",
         "content": {
             "hitPolicy": "first",
             "inputs": [{"id": "i1", "field": "importe", "name": "Importe"}],
             "outputs": [{"id": "o1", "field": "decision", "name": "Decision"}],
             "rules": [
                 {"i1": "< 500", "_id": "r1", "o1": "aprobar"},
                 {"i1": ">= 500", "_id": "r2", "o1": "revisar"},
             ],
         },
         "position": {"x": 200, "y": 0}},
        {"id": "out", "type": "outputNode", "name": "Response",
         "content": {"fields": [{"id": "decision", "field": "decision"}]},
         "position": {"x": 400, "y": 0}},
    ],
    "edges": [
        {"id": "e1", "sourceId": "input", "targetId": "table"},
        {"id": "e2", "sourceId": "table", "targetId": "out"},
    ],
}
engine = ZenEngine()
decision = engine.create_decision(json.dumps(graph))
print(decision.evaluate(json.dumps({"importe": 200})))

## 5. Benchmark: ¿compensa un motor?

| Motor | Tiempo/pago |
|---|---|
| `if/else` a mano | ~0,1 µs |
| GoRules ZEN | ~54 µs |
| CLIPS (`clipspy`) | ~96 µs |
| `experta` | ~190 µs |

**Actividad:** mide con `timeit` tu `if/else` frente a `rule-engine` para el mismo caso y calcula el factor.

In [ ]:
import timeit
ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}

def ifelse(c):
    if c["importe"] < 500 and c["antiguedad_meses"] >= 6 and c["n_intentos"] < 4:
        return "aprobar"
    if c["n_intentos"] >= 4 or c["antiguedad_meses"] < 6:
        return "rechazar"
    return "revisar"

print("if/else:", round(timeit.timeit(lambda: ifelse(ctx), number=100000) / 100000 * 1e6, 3), "us")